# Load dependencies

In [ ]:
#| default_exp datasets

In [ ]:
#|export
import math, torch
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from operator import itemgetter
from torch.utils.data import default_collate, DataLoader
from torch import tensor

In [ ]:
import torchvision.transforms.functional as TF
from datasets import load_dataset,load_dataset_builder

# Load data from HF hub

In [ ]:
name = "fashion_mnist"
ds_builder = load_dataset_builder(name)
dsd = load_dataset(name)

# Inspect data

In [ ]:
print(ds_builder.info.description)
print("====================")
print(ds_builder.info.features)
print("====================")
for split in ds_builder.info.splits.items(): print(split)
print("====================")
print(dsd)
print("====================")
train,test = dsd['train'],dsd['test']
x,y = ds_builder.info.features
print(x,y)
print("====================")
img = train[0][x]
label = train[0][y]
label_encoder = train.features[y].int2str
mpl.rcParams['image.cmap'] = 'gray'
plt.figure(figsize=(3,3))
plt.imshow(img)
plt.title(label_encoder(label));

# Define data collator

## 1st option

In [ ]:
def collate_fn(b):
    return {
        x:torch.stack([TF.to_tensor(o[x]) for o in b]),
        y:tensor([o[y] for o in b])
    }

dl = DataLoader(train, collate_fn=collate_fn, batch_size=16)
b = next(iter(dl))
for (k,v) in b.items():
    print(f"{k} | shape: {v.shape}")

## 2nd option

In [ ]:
def transforms(b):
    # this function actually receives a dict of batches
    # and the length of the batch is determined by the Dataloader batch_size!!! 
    b[x] = [TF.to_tensor(o) for o in b[x]]
    return b

tds = train.with_transform(transforms)
ds_batch = next(iter(tds))
print("dataset first item:")
for (k,v) in ds_batch.items():
    if isinstance(v, torch.Tensor): print(f"{k} | shape: {v.shape}")
    else: print(f"{k} | value: {v}")
print("====================")
dl = DataLoader(tds, batch_size=16) # torch.utils.data.default_collate will be used
b = next(iter(dl))
print("data loader first item:")
for (k,v) in b.items():
    print(f"{k} | shape: {v.shape}")
print("====================")

### Using decorators to write one-liner functions

In [ ]:
#|export
def inplace(f):
    def _f(b):
        f(b)
        return b
    return _f

In [ ]:
@inplace
def transformi(b): b[x] = [TF.to_tensor(o) for o in b[x]]
# def transformi(b): b[x] = [torch.flatten(TF.to_tensor(o)) for o in b[x]]
tdsf = train.with_transform(transformi)
dl = DataLoader(tdsf, batch_size=16)
b = next(iter(dl))
for (k,v) in b.items():
    print(f"{k} | shape: {v.shape}")
print("====================")

### Use `itemgetter` and `torch.utils.data.default_collate` to return tuple instead of dict
- HF datasets commonly work with dicts
- torch pipelines prefer tuples

#### examples of how they work

In [ ]:
d = dict(a=1,b=2,c=3)
ig = itemgetter('a','c')
print(ig(d))
print("=============")
class D:
    def __getitem__(self, k): return 1 if k=='a' else 2 if k=='b' else 3
d = D()
print(ig(d))
print("=============")
batch = dict(a=[1],b=[2]), dict(a=[3],b=[4])
print(default_collate(batch))
print("=============")

#### implementation

In [ ]:
#|export
def collate_dict(ds):
    get = itemgetter(*ds.features)
    def _f(b): return get(default_collate(b))
    return _f

In [ ]:
dlf = DataLoader(tdsf, batch_size=8, collate_fn=collate_dict(tdsf))
b = next(iter(dlf))
print([x.shape for x in b])

## Plotting images

In [ ]:
xb, yb = next(iter(dlf))

In [ ]:
#|export
def show_image(im, ax=None, figsize=None, title=None, noframe=True):
    "Show a PIL or PyTorch image on `ax`."
    if all(hasattr(im,attr) for attr in ('cpu','permute','detach')):
        im = im.detach().cpu()
        if len(im.shape)==3 and im.shape[0]<5: im=im.permute(1,2,0)
    elif not isinstance(im,np.ndarray): im=np.array(im)
    if im.shape[-1]==1: im=im[...,0]
    if ax is None: _,ax = plt.subplots(figsize=figsize)
    ax.imshow(im)
    if title is not None: ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([]) 
    if noframe: ax.axis('off')
    return ax

In [ ]:
fig,axs = plt.subplots(1,2)
show_image(xb[0], axs[0])
show_image(xb[1], axs[1]);

In [ ]:
#|export
def subplots(
    nrows:int=1, # Number of rows in returned axes grid
    ncols:int=1, # Number of columns in returned axes grid
    figsize:tuple=None, # Width, height in inches of the returned figure
    imsize:int=3, # Size (in inches) of images that will be displayed in the returned figure
    suptitle:str=None, # Title to be set to returned figure
): # fig and axs
    "A figure and set of subplots to display images of `imsize` inches"
    if figsize is None: figsize=(ncols*imsize, nrows*imsize)
    fig,ax = plt.subplots(nrows, ncols, figsize=figsize)
    if suptitle is not None: fig.suptitle(suptitle)
    if nrows*ncols==1: ax = np.array([ax])
    return fig,ax

In [ ]:
fig,axs = subplots(3,3, imsize=1)
imgs = xb[:8]
for ax,img in zip(axs.flat,imgs): show_image(img, ax)

In [ ]:
#|export
def get_grid(
    n:int, # Number of axes
    nrows:int=None, # Number of rows, defaulting to `int(math.sqrt(n))`
    ncols:int=None, # Number of columns, defaulting to `ceil(n/rows)`
    title:str=None, # If passed, title set to the figure
    weight:str='bold', # Title font weight
    size:int=14, # Title font size
    **kwargs,
): # fig and axs
    "Return a grid of `n` axes, `rows` by `cols`"
    if nrows: ncols = ncols or int(np.ceil(n/nrows))
    elif ncols: nrows = nrows or int(np.ceil(n/ncols))
    else:
        nrows = int(math.sqrt(n))
        ncols = int(np.ceil(n/nrows))
    fig,axs = subplots(nrows, ncols, **kwargs)
    for i in range(n, nrows*ncols): axs.flat[i].set_axis_off()
    if title is not None: fig.suptitle(title, weight=weight, size=size)
    return fig,axs

In [ ]:
fig,axs = get_grid(8, nrows=3, imsize=1)
for ax,img in zip(axs.flat,imgs): show_image(img, ax)

In [ ]:
#|export
def show_images(ims:list, # Images to show
                nrows:int|None=None, # Number of rows in grid
                ncols:int|None=None, # Number of columns in grid (auto-calculated if None)
                titles:list|None=None, # Optional list of titles for each image
                **kwargs):
    if titles: assert len(ims) == len(titles), "the number of images and titles must be the same"
    "Show all images `ims` as subplots with `rows` using `titles`"
    axs = get_grid(len(ims), nrows, ncols, **kwargs)[1].flat[:len(ims)]
    for im,t,ax in zip(ims, titles or [], axs): show_image(im, ax=ax, title=t)

In [ ]:
lbls = yb[:8]
print(lbls)
names = "Top Trouser Pullover Dress Coat Sandal Shirt Sneaker Bag Boot".split()
titles = itemgetter(*lbls)(names)
print(' '.join(titles))
show_images(imgs, imsize=1.7, titles=titles, nrows=3)